# 🎵 Mood Classifier — DeBERTa-v3-small (9 emotions) v3

**Target:** 87–92% validation accuracy

**Changes from v2:**
- `distilbert-base-uncased` → `microsoft/deberta-v3-small` (+5–8% accuracy)
- Added **EmpatheticDialogues** (Facebook, 25k natural conversations)
- Added **ISEAR** dataset (7.5k formal emotion sentences)
- **Synthetic data** generation — 50 sentences per class via templates
- Two-phase training (freeze → unfreeze)
- Confidence threshold: if top score < 40%, return top-2 moods

**Runtime:** T4 GPU — Runtime → Change runtime type → T4 GPU  
**Estimated time:** ~25–35 min total

## 0 — Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')
    raise SystemExit('GPU required')

## 1 — Install dependencies

In [ ]:
# DeBERTa needs sentencepiece + protobuf
!pip install -q transformers datasets torch scikit-learn \
    optimum[onnxruntime] onnx onnxruntime \
    sentencepiece protobuf

## 2 — Label definitions

In [ ]:
import pandas as pd
import numpy as np
import json, os, re

OUR_LABELS = ['happy','sad','angry','calm','anxious','focused','loved','melancholic','confident']
LABEL2ID   = {l: i for i, l in enumerate(OUR_LABELS)}
ID2LABEL   = {i: l for l, i in LABEL2ID.items()}
NUM_LABELS = len(OUR_LABELS)
print('Labels:', LABEL2ID)

## 3 — Source 1: GoEmotions

In [ ]:
from datasets import load_dataset

GO_EMOTIONS_LABELS = [
    'admiration','amusement','anger','annoyance','approval','caring','confusion',
    'curiosity','desire','disappointment','disapproval','disgust','embarrassment',
    'excitement','fear','gratitude','grief','joy','love','nervousness','optimism',
    'pride','realization','relief','remorse','sadness','surprise','neutral'
]

# Tight mapping only — removed noisy ones
GE_TO_MOOD = {
    'amusement':     'happy',
    'anger':         'angry',
    'annoyance':     'angry',
    'caring':        'loved',
    'desire':        'loved',
    'disappointment':'melancholic',
    'disapproval':   'angry',
    'disgust':       'angry',
    'embarrassment': 'anxious',
    'excitement':    'happy',
    'fear':          'anxious',
    'gratitude':     'loved',
    'grief':         'melancholic',
    'joy':           'happy',
    'love':          'loved',
    'nervousness':   'anxious',
    'optimism':      'happy',
    'pride':         'confident',
    'relief':        'calm',
    'remorse':       'sad',
    'sadness':       'sad',
}

print('Loading GoEmotions...')
ge = load_dataset('google-research-datasets/go_emotions', 'simplified', trust_remote_code=True)
rows = []
for split in ['train', 'validation', 'test']:
    for ex in ge[split]:
        text = ex['text'].strip()
        if not text or not ex['labels']:
            continue
        ge_label = GO_EMOTIONS_LABELS[ex['labels'][0]] if ex['labels'][0] < len(GO_EMOTIONS_LABELS) else None
        mood = GE_TO_MOOD.get(ge_label)
        if mood:
            rows.append({'text': text, 'label': mood, 'source': 'goemotions'})

df_ge = pd.DataFrame(rows)
print(f'GoEmotions: {len(df_ge)} rows')
print(df_ge['label'].value_counts())

## 4 — Source 2: EmpatheticDialogues (Facebook)

25k empathetic conversations with 32 emotion labels. Covers natural, conversational language
that GoEmotions (Reddit) misses.

In [ ]:
# EmpatheticDialogues emotion → our mood
ED_TO_MOOD = {
    'joyful':        'happy',
    'excited':       'happy',
    'content':       'happy',
    'optimistic':    'happy',
    'proud':         'confident',
    'confident':     'confident',
    'grateful':      'loved',
    'caring':        'loved',
    'trusting':      'loved',
    'sentimental':   'melancholic',
    'nostalgic':     'melancholic',
    'lonely':        'sad',
    'sad':           'sad',
    'devastated':    'sad',
    'guilty':        'sad',
    'ashamed':       'sad',
    'angry':         'angry',
    'furious':       'angry',
    'disgusted':     'angry',
    'jealous':       'angry',
    'afraid':        'anxious',
    'anxious':       'anxious',
    'terrified':     'anxious',
    'embarrassed':   'anxious',
    'apprehensive':  'anxious',
    'anticipating':  'focused',
    'prepared':      'focused',
    'impressed':     'focused',
    'surprised':     None,
    'faithful':      None,
    'hopeful':       None,
    'disappointed':  'melancholic',
}

# facebook/empathetic_dialogues uses old dataset scripts blocked in datasets>=2.15
# Load directly from parquet files instead
print('Loading EmpatheticDialogues (parquet)...')
try:
    ed = load_dataset(
        'parquet',
        data_files={
            'train':      'hf://datasets/datasets-community/empathetic_dialogues/data/train-*.parquet',
            'validation': 'hf://datasets/datasets-community/empathetic_dialogues/data/validation-*.parquet',
            'test':       'hf://datasets/datasets-community/empathetic_dialogues/data/test-*.parquet',
        }
    )
    print('Columns:', ed['train'].column_names)

    rows_ed = []
    seen = set()
    for split in ed.keys():
        for ex in ed[split]:
            # column names vary — try all common variants
            text    = (ex.get('prompt') or ex.get('utterance') or ex.get('text') or '').strip()
            emotion = (ex.get('context') or ex.get('emotion') or ex.get('label') or '').lower().strip()
            if not text or text in seen or len(text) < 15:
                continue
            seen.add(text)
            mood = ED_TO_MOOD.get(emotion)
            if mood:
                rows_ed.append({'text': text, 'label': mood, 'source': 'empathetic'})

    df_ed = pd.DataFrame(rows_ed)
    if df_ed.empty:
        print('⚠️  0 rows loaded — check column names above and update ED_TO_MOOD keys if needed')
    else:
        print(f'EmpatheticDialogues: {len(df_ed)} rows')
        print(df_ed['label'].value_counts())

except Exception as e:
    print(f'⚠️  EmpatheticDialogues failed ({e}) — skipping, training will still work without it')
    df_ed = pd.DataFrame(columns=['text', 'label', 'source'])

## 5 — Source 3: ISEAR

7,500 sentences from a psychology study — respondents described situations that caused specific emotions.
Formal, diverse language. Great for calm, focused, confident which Reddit underrepresents.

In [ ]:
# ISEAR maps 7 emotions: joy, fear, anger, sadness, disgust, shame, guilt
ISEAR_TO_MOOD = {
    'joy':     'happy',
    'fear':    'anxious',
    'anger':   'angry',
    'sadness': 'sad',
    'disgust': 'angry',
    'shame':   'sad',
    'guilt':   'melancholic',
}

print('Loading ISEAR...')
try:
    isear = load_dataset('ywkim/isear', trust_remote_code=True)
    rows_isear = []
    for split in isear.keys():
        for ex in isear[split]:
            text  = (ex.get('text') or ex.get('sentence') or '').strip()
            label = (ex.get('label') or ex.get('emotion') or '').lower().strip()
            mood  = ISEAR_TO_MOOD.get(label)
            if mood and text and len(text) > 10:
                rows_isear.append({'text': text, 'label': mood, 'source': 'isear'})
    df_isear = pd.DataFrame(rows_isear)
    print(f'ISEAR: {len(df_isear)} rows')
    print(df_isear['label'].value_counts())
except Exception as e:
    print(f'ISEAR load failed ({e}) — skipping, still fine without it')
    df_isear = pd.DataFrame(columns=['text','label','source'])

## 6 — Source 4: Seed sentences (hand-crafted)

In [ ]:
SEED_DATA = {
    'happy': [
        "I'm so happy today, everything feels perfect",
        "Just got amazing news, feeling on top of the world",
        "Life is wonderful and I feel so grateful",
        "Had the best day ever, can't stop smiling",
        "Feeling absolutely joyful and full of energy",
        "Everything is going great, I'm thrilled",
        "I woke up feeling fantastic and ready to take on the day",
        "So excited about what's coming next",
        "Pure happiness, couldn't ask for more",
        "Today was brilliant, I feel amazing",
        "Bursting with joy, life couldn't be better",
        "I'm on cloud nine right now",
        "Good vibes only, feeling so upbeat",
        "Just celebrated and feeling over the moon",
        "Everything clicked today and I'm beaming",
        "So much positivity flowing through me right now",
        "Can't stop laughing, today has been incredible",
        "Feeling light and free and genuinely happy",
        "My heart is full of joy",
        "Absolutely thrilled with how things turned out",
    ],
    'sad': [
        "I've been crying all day and don't know why",
        "Feeling really down and hopeless lately",
        "Missing someone so much it hurts",
        "Everything feels empty and pointless",
        "I just want to stay in bed and not face the world",
        "So heartbroken I can barely function",
        "Nobody seems to understand what I'm going through",
        "Lost and sad, nothing feels okay",
        "I feel a deep sadness I can't shake",
        "Tears keep coming and I don't know how to stop them",
        "I feel like nothing matters anymore",
        "So much grief weighing on me lately",
        "I'm devastated and don't know how to move on",
        "The loneliness is unbearable right now",
        "Feeling completely broken and lost",
        "The sadness just won't lift no matter what I do",
        "I feel invisible and deeply unloved",
        "Can't find a reason to feel okay today",
        "Everything feels grey and heavy",
        "I keep crying for no reason and feel hollow inside",
    ],
    'angry': [
        "I'm absolutely furious, this is unacceptable",
        "So frustrated I could scream right now",
        "They did it again and I'm done tolerating it",
        "Rage is the only word for how I feel",
        "Beyond angry, this situation is infuriating",
        "Can't believe how disrespectful people can be",
        "I'm seething and need to vent",
        "Totally livid about what happened today",
        "I'm done being patient, I'm outraged",
        "My blood is boiling and I can't calm down",
        "Furious doesn't even begin to cover it",
        "So irritated I can barely speak right now",
        "That was the last straw, I'm completely fed up",
        "I want to punch a wall I'm so frustrated",
        "Pure rage, I've never been this mad",
        "Disgusted and angry at how things went down",
        "How dare they, I'm absolutely mad",
        "Unbelievably angry at the whole situation",
        "I hate when people treat others like that",
        "This injustice makes me so angry I can't think straight",
    ],
    'calm': [
        "Feeling really peaceful and at ease today",
        "Just took a long walk and feel completely relaxed",
        "Nothing is bothering me, I feel serene",
        "In a gentle, quiet headspace right now",
        "Meditating has brought me so much calm",
        "Everything feels still and okay",
        "Totally at peace with where I am in life",
        "Soft music and tea, feeling tranquil",
        "No rush, no stress, just calm",
        "Breathing deeply and feeling grounded",
        "A quiet Sunday morning kind of feeling",
        "Nothing on my mind, just stillness",
        "I feel centred and unhurried today",
        "The world feels gentle right now",
        "Just finished yoga and feel so at ease",
        "Completely unbothered and in a peaceful state",
        "Letting everything go and just being present",
        "I feel like a still lake, undisturbed",
        "Everything is okay and I feel safe and quiet",
        "No anxiety, no rush, just a soft peaceful feeling",
        "Feeling mellow and easy, nothing to worry about",
        "A deep sense of ease has settled over me",
        "Warm, slow, and tranquil — that's today",
        "Just sitting quietly and it feels wonderful",
        "The calmness is almost meditative today",
    ],
    'anxious': [
        "My mind won't stop racing, I'm so anxious",
        "Feeling overwhelmed by everything on my plate",
        "Can't shake this nervousness before the big day",
        "My heart is pounding and I don't know why",
        "Stressed out beyond measure, too much happening at once",
        "Worried about everything and nothing at the same time",
        "The uncertainty is making me spiral",
        "I feel tense all over and can't relax",
        "Panic is creeping in and I can't calm down",
        "So much to do and I don't know where to start",
        "My thoughts are spiralling and I can't stop them",
        "I feel like something bad is about to happen",
        "Constant low-level dread that won't go away",
        "My chest feels tight from all this stress",
        "Too many things at once, I'm about to break",
        "Nervous energy I can't shake no matter what",
        "I keep worrying about things I can't control",
        "Feel like I'm on the edge of a panic attack",
        "Everything feels like a threat right now",
        "Overthinking every little thing and can't stop",
    ],
    'focused': [
        "Deep in work mode, totally in the zone",
        "Locked in on this task, hours have flown by",
        "My mind is sharp and I'm crushing this project",
        "Zero distractions, completely focused",
        "Productivity is through the roof today",
        "Studying hard, need something to keep the momentum",
        "Coding for hours and feeling fully engaged",
        "Need to concentrate, looking for focus music",
        "In a flow state, everything clicking",
        "Head down, grinding through this work session",
        "Completely absorbed in what I'm building",
        "Nothing exists right now except this task",
        "Hyper-focused and making serious progress",
        "Deep work session, don't disturb",
        "Every thought is directed at the problem in front of me",
        "I've been at my desk for hours and haven't noticed",
        "Locked in, dialled in, ready to ship",
        "Full concentration, everything else fades away",
        "Reading deeply and taking notes, totally engaged",
        "My attention is laser-sharp right now",
        "Can't be pulled away from this, too focused",
        "Work is flowing effortlessly, I'm in the zone",
        "Three hours of deep focus, no interruptions",
        "Solving hard problems and loving every second",
        "Nothing but the task in front of me matters right now",
    ],
    'loved': [
        "I feel so loved and appreciated by the people around me",
        "My heart is full, surrounded by people who care",
        "That hug meant everything, feeling so warm inside",
        "Being in love is the best feeling in the world",
        "I feel cherished and safe with this person",
        "So much gratitude for the love I have in my life",
        "Feeling deeply connected and cared for",
        "Love is in the air and I am soaking it all in",
        "My partner made me feel so special today",
        "Warmth and affection all around me right now",
        "Someone told me they loved me today and I melted",
        "I feel seen, heard, and truly cared about",
        "So much tenderness in my life right now",
        "Every little gesture today reminded me I'm loved",
        "My heart is overflowing with love",
        "I've never felt so understood and cared for",
        "The people in my life make me feel so valued",
        "Wrapped in so much love I could cry",
        "Feeling adored and it's the most beautiful thing",
        "Pure love, no conditions, just warmth",
        "I am so grateful for the people who show up for me",
        "Holding someone close and feeling completely at home",
        "I feel safe and cherished in a way I've longed for",
        "The love in this moment is completely overwhelming",
        "Surrounded by warmth and genuine affection",
    ],
    'melancholic': [
        # Original 25
        "Feeling nostalgic for a time that will never come back",
        "A bittersweet sadness has settled over me",
        "Thinking about the past and feeling wistful",
        "There's a quiet ache I can't quite name",
        "Melancholy creeping in as I watch the rain",
        "Things ended and I'm still not over it",
        "A deep longing for something lost",
        "Beauty and sadness mixed together tonight",
        "Reflective and a little blue, like a cloudy afternoon",
        "Something about today feels heavy and tender",
        "Thinking about old friends I've drifted from",
        "There's a softness to this sadness, not quite grief",
        "Listening to old songs and feeling time slipping",
        "The past keeps calling and I can't stop looking back",
        "A hollow sweetness, like something precious that's gone",
        "Staring at old photos and feeling that familiar ache",
        "Not sad exactly, but not quite okay either — wistful",
        "Something beautiful ended and I'm still sitting with it",
        "A tender, quiet kind of grief I carry gently",
        "Longing for something I can't quite put into words",
        "Bittersweet memories keep surfacing today",
        "Everything reminds me of something I've lost",
        "The kind of sadness that feels almost like love",
        "I miss who I used to be, and who we used to be",
        "The feeling of autumn afternoons — beautiful and fleeting",
        # 40 new — clearly wistful/nostalgic, NOT sad
        "Driving past my old school and smiling at the memories",
        "That song came on and suddenly I was seventeen again",
        "Found an old photo and couldn't stop looking at it",
        "The smell of rain on summer pavement takes me back every time",
        "Watching a sunset and thinking about how fast time moves",
        "I love this city but it holds so many ghosts for me",
        "Scrolling through old texts and feeling that familiar warmth",
        "The way autumn light falls always makes me feel something unnamed",
        "Old movies hit different — they carry a whole era with them",
        "I miss the version of me who didn't know what was coming",
        "There's a specific ache in revisiting places from your past",
        "The old neighbourhood looks the same but feels completely different",
        "Childhood summers felt infinite in a way nothing does now",
        "Reading my old journal entries felt like meeting a stranger",
        "That café closed down and I keep forgetting until I walk past",
        "I still know every word to songs I haven't heard in a decade",
        "Looking at old group photos and wondering where everyone went",
        "There's something bittersweet about outgrowing a phase of your life",
        "The version of this city I fell in love with is mostly gone now",
        "I collect moments like these — beautiful and already fading",
        "Fond and a little wistful, the way you feel after a good dream",
        "Everything is fine, I'm just thinking about how things used to be",
        "Not unhappy — just tender about the passage of time",
        "A quiet kind of longing that feels almost sweet",
        "Missing something I can't quite name — a feeling more than a place",
        "Bittersweet is the only word for this particular kind of okay",
        "I'm happy now but I still carry the past gently with me",
        "Grateful for what was, even if it's over",
        "Some memories are just too good not to feel wistful about",
        "There's beauty in missing things — it means they mattered",
        "I'm doing well but the nostalgia hits hard sometimes",
        "Not sad, just reflective — there's a difference",
        "Life is good, I just find myself looking back a lot today",
        "Content but wistful — two things that can coexist",
        "I don't want to go back, I just want to remember it better",
        "Missing people who are still alive but drifted away",
        "The bittersweetness of a chapter closing even when the next is good",
        "Sitting with the ache of good things ending rather than bad",
        "Nostalgic for moments I knew were precious even as they happened",
        "There's a softness to this feeling — nothing sharp, just tender",
    ],
    'confident': [
        "I know exactly what I'm doing and I'm owning it",
        "Feeling strong, capable, and unstoppable",
        "I walked in and commanded the room",
        "No doubt in my mind, I've got this",
        "Self-assured and ready to take on any challenge",
        "I trust myself completely right now",
        "Radiating confidence and clarity today",
        "Nothing can shake me, I'm solid",
        "I presented with power and everyone noticed",
        "Bold, decisive, and fully in control",
        "I know my worth and I'm not settling",
        "Walking with my head high, owning every step",
        "I showed up and delivered, no hesitation",
        "My decisions are clear and I stand behind them",
        "Nothing intimidates me today, I feel invincible",
        "I spoke up and it felt powerful",
        "Every move I make today comes from a place of certainty",
        "I've worked hard for this and I deserve it",
        "Totally in command of myself and the situation",
        "Unshakeable today — whatever comes, I'm ready",
        "I feel magnetic and powerful right now",
        "My confidence is quiet but absolute",
        "I walked away from that knowing I nailed it",
        "Grounded, clear, and completely self-assured",
        "I am exactly where I need to be and I know it",
    ],
}

seed_rows = []
for mood, texts in SEED_DATA.items():
    for t in texts:
        seed_rows.append({'text': t, 'label': mood, 'source': 'seed'})

df_seed = pd.DataFrame(seed_rows * 8)
print(f'Seed rows (×8): {len(df_seed)}')
print(pd.Series([r["label"] for r in seed_rows]).value_counts())

## 7 — Source 5: Synthetic data

Template-based generation. Fills gaps GoEmotions can't cover (calm, focused, confident).
No API needed — pure Python string templates.

In [ ]:
import random
random.seed(42)

# Each mood has: sentence starters × endings — cross-product gives N×M unique sentences
SYNTHETIC_TEMPLATES = {
    'happy': {
        'starters': [
            "I'm feeling", "Today I feel", "Right now I feel", "I woke up feeling",
            "Can't stop feeling", "Everything makes me feel",
        ],
        'endings': [
            "so happy and alive", "amazing and full of energy", "like everything is wonderful",
            "joyful and light", "absolutely fantastic", "thrilled and grateful",
            "excited about everything", "on top of the world", "beaming with happiness",
        ],
    },
    'sad': {
        'starters': [
            "I'm feeling", "I can't help feeling", "Everything feels", "I've been feeling",
            "Right now I feel", "I just feel",
        ],
        'endings': [
            "so sad and empty", "hopeless and lost", "heartbroken and alone",
            "deeply unhappy", "like nothing matters", "devastated and broken",
            "heavy and numb", "tearful and low", "hollow inside",
        ],
    },
    'angry': {
        'starters': [
            "I'm feeling", "I can't believe how", "I am absolutely", "Right now I feel",
            "I'm so", "Everything is making me feel",
        ],
        'endings': [
            "furious and livid", "angry and fed up", "enraged and frustrated",
            "infuriated by everything", "so angry I can't think straight", "boiling with rage",
            "beyond irritated", "outraged and disgusted", "ready to explode",
        ],
    },
    'calm': {
        'starters': [
            "I'm feeling", "Right now I feel", "Everything feels", "I feel so",
            "Today I feel", "I woke up feeling",
        ],
        'endings': [
            "calm and at peace", "completely relaxed and serene", "tranquil and grounded",
            "still and unhurried", "peaceful and centred", "quiet and content",
            "at ease with everything", "gentle and slow", "completely undisturbed",
        ],
    },
    'anxious': {
        'starters': [
            "I'm feeling", "I can't stop feeling", "My mind is", "I keep feeling",
            "Right now I feel", "Everything is making me feel",
        ],
        'endings': [
            "anxious and overwhelmed", "stressed and panicked", "nervous and tense",
            "worried about everything", "on edge and restless", "spiralling with anxiety",
            "too stressed to function", "fearful and uneasy", "like something bad will happen",
        ],
    },
    'focused': {
        'starters': [
            "I'm feeling", "Right now I feel", "I'm in", "My mind is",
            "I've been", "I feel completely",
        ],
        'endings': [
            "completely focused and sharp", "locked in and productive", "deep in work mode",
            "fully concentrated on my task", "totally absorbed in what I'm doing",
            "in a perfect flow state", "dialled in and making progress",
            "zeroed in with no distractions", "mentally clear and on task",
        ],
    },
    'loved': {
        'starters': [
            "I'm feeling", "I feel so", "Right now I feel", "Everything makes me feel",
            "I've never felt so", "Today I feel",
        ],
        'endings': [
            "loved and cherished", "deeply cared for and appreciated", "warm and adored",
            "surrounded by so much love", "seen and truly understood", "safe and deeply loved",
            "full of warmth and affection", "held and cared for", "so grateful to be loved",
        ],
    },
    'melancholic': {
        'starters': [
            "I'm feeling", "There's something", "I feel", "Right now I feel",
            "I can't shake this feeling of", "Everything feels",
        ],
        'endings': [
            "wistful and nostalgic", "bittersweet and reflective", "a quiet ache inside",
            "heavy with longing", "tender and a little sad", "like something beautiful is fading",
            "like I'm missing something I can't name", "soft and sad in a gentle way",
            "deeply nostalgic for the past",
        ],
    },
    'confident': {
        'starters': [
            "I'm feeling", "Right now I feel", "I feel so", "Today I feel",
            "I woke up feeling", "Everything makes me feel",
        ],
        'endings': [
            "confident and unstoppable", "strong and self-assured", "bold and decisive",
            "completely in control", "powerful and capable", "certain and grounded",
            "like I can handle anything", "ready to take on the world", "clear and unshakeable",
        ],
    },
}

synth_rows = []
for mood, tmpl in SYNTHETIC_TEMPLATES.items():
    for starter in tmpl['starters']:
        for ending in tmpl['endings']:
            synth_rows.append({
                'text':   f"{starter} {ending}.",
                'label':  mood,
                'source': 'synthetic'
            })

df_synth = pd.DataFrame(synth_rows * 4)  # repeat 4x
print(f'Synthetic rows (×4): {len(df_synth)}')
print(pd.DataFrame(synth_rows)['label'].value_counts())

## 8 — Combine, balance & split

In [ ]:
from sklearn.model_selection import train_test_split

df_all = pd.concat([df_ge, df_ed, df_isear, df_seed, df_synth], ignore_index=True)
df_all = df_all.dropna(subset=['text', 'label'])
df_all = df_all[df_all['label'].isin(OUR_LABELS)]
df_all['label_id'] = df_all['label'].map(LABEL2ID)

print('Raw combined distribution:')
print(df_all['label'].value_counts())

# Cap at 3000 per class — balanced training
df_balanced = (
    df_all.groupby('label', group_keys=False)
          .apply(lambda g: g.sample(min(len(g), 3000), random_state=42))
          .sample(frac=1, random_state=42)
          .reset_index(drop=True)
)

print(f'\nBalanced distribution (cap 3000):')
print(df_balanced['label'].value_counts())
print(f'\nTotal: {len(df_balanced)} examples')

# Source breakdown
print('\nSource breakdown:')
print(df_balanced['source'].value_counts())

# 90/10 stratified split
train_df, val_df = train_test_split(
    df_balanced, test_size=0.1, stratify=df_balanced['label'], random_state=42
)
print(f'\nTrain: {len(train_df)}  |  Val: {len(val_df)}')

## 9 — Tokenize

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

# DeBERTa-v3-small — better than DistilBERT, same size as BERT-base
MODEL_NAME = 'microsoft/deberta-v3-small'
MAX_LEN    = 128
BATCH_SIZE = 32  # DeBERTa uses more memory than DistilBERT

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MoodDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(
            list(texts),
            truncation=True,
            padding='max_length',
            max_length=MAX_LEN,
            return_tensors='pt'
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self): return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = self.labels[i]
        return item

train_ds = MoodDataset(train_df['text'].values, train_df['label_id'].values)
val_ds   = MoodDataset(val_df['text'].values,   val_df['label_id'].values)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print('Datasets ready.')

## 10 — Two-phase training

**Phase 1 (2 epochs):** Freeze transformer body, train only the classification head.
This warms up the head without corrupting pretrained weights.

**Phase 2 (5 epochs):** Unfreeze everything, fine-tune end-to-end with a lower LR.
Best checkpoint is saved and restored automatically.

In [ ]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
import torch.nn as nn
import time

device = torch.device('cuda')

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
).to(device)

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

def evaluate(loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['labels'].to(device)
            out  = model(input_ids=ids, attention_mask=mask)
            correct += (out.logits.argmax(-1) == lbls).sum().item()
            total   += lbls.size(0)
    return correct / total

def run_phase(phase_name, epochs, lr, frozen=False):
    global best_val_acc

    # Freeze / unfreeze transformer body
    for name, param in model.named_parameters():
        if 'classifier' in name or 'pooler' in name:
            param.requires_grad = True
        else:
            param.requires_grad = not frozen

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n── {phase_name} | lr={lr} | trainable params: {trainable:,}')
    print('-' * 60)

    optimizer  = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler  = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, int(total_steps * 0.06)),
        num_training_steps=total_steps
    )

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        t0 = time.time()

        for batch in train_loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['labels'].to(device)
            out  = model(input_ids=ids, attention_mask=mask)
            loss = loss_fn(out.logits, lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += loss.item()

        val_acc  = evaluate(val_loader)
        avg_loss = total_loss / len(train_loader)
        elapsed  = time.time() - t0

        flag = ''
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), '/content/best_model.pt')
            flag = ' ← best'

        print(f'  Epoch {epoch}/{epochs} | loss {avg_loss:.4f} | val {val_acc:.3f} | {elapsed:.0f}s{flag}')

best_val_acc = 0.0

# Phase 1 — head only
run_phase('Phase 1 — head only (warm-up)', epochs=2, lr=1e-3, frozen=True)

# Phase 2 — full model
run_phase('Phase 2 — full fine-tune', epochs=5, lr=2e-5, frozen=False)

# Restore best checkpoint
model.load_state_dict(torch.load('/content/best_model.pt'))
print(f'\nRestored best checkpoint. Val acc: {best_val_acc:.3f}')

## 11 — Per-class report

In [ ]:
from sklearn.metrics import classification_report

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        out  = model(input_ids=ids, attention_mask=mask)
        all_preds.extend(out.logits.argmax(-1).cpu().tolist())
        all_labels.extend(batch['labels'].tolist())

print(classification_report(all_labels, all_preds, target_names=OUR_LABELS, digits=3))
print('\n💡 Any class F1 < 0.70 → add more seed/synthetic sentences for it and retrain Phase 2 only.')

## 11b — Fix melancholic: inject better examples + retrain

In [ ]:
MELANCHOLIC_EXTRA = [
    # Clearly nostalgic/wistful — no sad/cry/hurt vocabulary
    "Driving past my old school and smiling at the memories",
    "That song came on and suddenly I was seventeen again",
    "Found an old photo and couldn't stop looking at it",
    "The smell of rain on summer pavement takes me back every time",
    "Watching a sunset and thinking about how fast time moves",
    "I love this city but it holds so many ghosts for me",
    "Scrolling through old texts and feeling that familiar warmth",
    "The way autumn light falls always makes me feel something unnamed",
    "Old movies hit different — they carry a whole era with them",
    "I miss the version of me who didn't know what was coming",
    "There's a specific ache in revisiting places from your past",
    "The old neighbourhood looks the same but feels completely different",
    "Childhood summers felt infinite in a way nothing does now",
    "Reading my old journal entries felt like meeting a stranger",
    "That café closed down and I keep forgetting until I walk past",
    "I still know every word to songs I haven't heard in a decade",
    "Looking at old group photos and wondering where everyone went",
    "There's something bittersweet about outgrowing a phase of your life",
    "The version of this city I fell in love with is mostly gone now",
    "I collect moments like these — beautiful and already fading",
    # Wistful without being sad
    "Fond and a little wistful, the way you feel after a good dream",
    "Everything is fine, I'm just thinking about how things used to be",
    "Not unhappy — just tender about the passage of time",
    "A quiet kind of longing that feels almost sweet",
    "Missing something I can't quite name — a feeling more than a place",
    "Bittersweet is the only word for this particular kind of okay",
    "I'm happy now but I still carry the past gently with me",
    "Grateful for what was, even if it's over",
    "Some memories are just too good not to feel wistful about",
    "There's beauty in missing things — it means they mattered",
    # Contrast with sad — explicitly NOT sad
    "I'm doing well but the nostalgia hits hard sometimes",
    "Not sad, just reflective — there's a difference",
    "Life is good, I just find myself looking back a lot today",
    "Content but wistful — two things that can coexist",
    "I don't want to go back, I just want to remember it better",
    "Missing people who are still alive but drifted away",
    "The bittersweetness of a chapter closing even when the next is good",
    "Sitting with the ache of good things ending rather than bad",
    "Nostalgic for moments I knew were precious even as they happened",
    "There's a softness to this feeling — nothing sharp, just tender",
]

# Inject into training data — repeat 15x for strong signal
extra_rows = [
    {'text': t, 'label': 'melancholic', 'label_id': LABEL2ID['melancholic']}
    for t in MELANCHOLIC_EXTRA
]
extra_df = pd.DataFrame(extra_rows * 15)

# Append to existing train split and reshuffle
train_df_fixed = pd.concat([train_df, extra_df], ignore_index=True).sample(frac=1, random_state=99)

# Rebuild train loader
train_ds_fixed = MoodDataset(train_df_fixed['text'].values, train_df_fixed['label_id'].values)
train_loader   = DataLoader(train_ds_fixed, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)

print(f'Train set: {len(train_df_fixed)} rows (+{len(extra_df)} melancholic examples)')

# Retrain from best checkpoint — 4 epochs at low LR
model.load_state_dict(torch.load('/content/best_model.pt'))
for param in model.parameters():
    param.requires_grad = True

loss_fn   = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = AdamW(model.parameters(), lr=6e-6, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=0,
    num_training_steps=len(train_loader) * 4
)

print(f'Retraining from val acc {best_val_acc:.3f} | lr=6e-6 | 4 epochs')
print('-' * 50)

for epoch in range(1, 5):
    model.train()
    total_loss = 0
    t0 = time.time()
    for batch in train_loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        loss = loss_fn(model(input_ids=ids, attention_mask=mask).logits, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        total_loss += loss.item()
    val_acc = evaluate(val_loader)
    flag = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/best_model.pt')
        flag = ' ← best'
    print(f'  Epoch {epoch}/4 | loss {total_loss/len(train_loader):.4f} | val {val_acc:.3f} | {time.time()-t0:.0f}s{flag}')

model.load_state_dict(torch.load('/content/best_model.pt'))
print(f'\nDone. Best val acc: {best_val_acc:.3f}')
print('Now re-run Step 11 to see the updated per-class report.')

## 12 — Sanity test + confidence threshold

In [ ]:
import torch.nn.functional as F

CONFIDENCE_THRESHOLD = 0.40  # below this → return top-2

def predict(text: str):
    model.eval()
    enc = tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=MAX_LEN, padding='max_length'
    ).to(device)
    with torch.no_grad():
        probs = F.softmax(model(**enc).logits, dim=-1)[0]
    sorted_ids = probs.argsort(descending=True)
    top_conf = probs[sorted_ids[0]].item()
    return {
        'mood':        ID2LABEL[sorted_ids[0].item()],
        'confidence':  round(top_conf * 100, 1),
        'uncertain':   top_conf < CONFIDENCE_THRESHOLD,
        'second_mood': ID2LABEL[sorted_ids[1].item()] if top_conf < CONFIDENCE_THRESHOLD else None,
        'all_scores':  {ID2LABEL[i]: round(probs[i].item() * 100, 1) for i in range(NUM_LABELS)}
    }

tests = [
    ('happy',       "I'm so happy today everything is amazing"),
    ('sad',         "Feeling really down and hopeless, can't stop crying"),
    ('angry',       "I'm furious about what happened, absolutely livid"),
    ('calm',        "Totally peaceful and relaxed, just breathing slowly"),
    ('anxious',     "So stressed and overwhelmed, my mind won't stop racing"),
    ('focused',     "Deep in work mode, totally in the zone, hours flying by"),
    ('loved',       "I feel so cherished and adored, my heart is full"),
    ('melancholic', "Nostalgic and wistful, missing something I can't name"),
    ('confident',   "I know exactly what I want and I'm going for it"),
    ('?',           "I don't really feel anything in particular today"),
    ('?',           "Not sad but not happy either, just kind of numb"),
    ('?',           "It was just a normal day, nothing special happened"),
]

print(f'{"Expected":>12}  {"Got":>12}  {"Conf":>6}  {"Uncertain":<10}  Text')
print('-' * 80)
correct = total_known = 0
for expected, text in tests:
    r = predict(text)
    if expected != '?':
        match = '✓' if r['mood'] == expected else '✗'
        if r['mood'] == expected: correct += 1
        total_known += 1
    else:
        match = ' '
    uncertain_str = f"→ also {r['second_mood']}" if r['uncertain'] else ''
    print(f'{expected:>12}  {r["mood"]:>12}  {r["confidence"]:>5}%  {uncertain_str:<18}  {text[:45]}')

print(f'\nSanity accuracy: {correct}/{total_known} = {correct/total_known:.0%}')

## 13 — Save HuggingFace model

In [ ]:
SAVE_DIR = '/content/mood_model_hf'
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open(f'{SAVE_DIR}/label_map.json', 'w') as f:
    json.dump({
        'id2label':             ID2LABEL,
        'label2id':             LABEL2ID,
        'confidence_threshold': CONFIDENCE_THRESHOLD
    }, f, indent=2)

print(f'Saved to {SAVE_DIR}')

## 14 — Export to ONNX

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import pipeline
import shutil

ONNX_DIR = '/content/mood_model_onnx'
os.makedirs(ONNX_DIR, exist_ok=True)

print('Exporting to ONNX...')
ort_model = ORTModelForSequenceClassification.from_pretrained(SAVE_DIR, export=True)
ort_model.save_pretrained(ONNX_DIR)
tokenizer.save_pretrained(ONNX_DIR)
shutil.copy(f'{SAVE_DIR}/label_map.json', f'{ONNX_DIR}/label_map.json')

# Verify
pipe = pipeline('text-classification', model=ort_model, tokenizer=tokenizer, top_k=3)
test_out = pipe('I feel absolutely amazing and full of energy today!')
print('ONNX test:', test_out)

# File sizes
total_mb = 0
for f in sorted(os.listdir(ONNX_DIR)):
    mb = os.path.getsize(os.path.join(ONNX_DIR, f)) / 1024 / 1024
    total_mb += mb
    print(f'  {f}: {mb:.1f} MB')
print(f'  Total: {total_mb:.1f} MB')

## 15 — Download

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/mood_model_onnx', 'zip', '/content', 'mood_model_onnx')
files.download('/content/mood_model_onnx.zip')
print('Download started.')

## ✅ Done!

**Expected results with v3:**

| Class | Expected F1 |
|-------|-------------|
| happy | 0.88–0.93 |
| sad | 0.85–0.91 |
| angry | 0.88–0.94 |
| calm | 0.82–0.89 |
| anxious | 0.83–0.90 |
| focused | 0.80–0.88 |
| loved | 0.83–0.90 |
| melancholic | 0.76–0.85 |
| confident | 0.80–0.88 |
| **overall** | **87–92%** |

**If a class is still below F1=0.70:**
Add more seed sentences for it in Step 6 and re-run Phase 2 only (Step 10 — change `run_phase` call to skip Phase 1).

**The `uncertain` flag:**  
When confidence < 40%, the API returns both `mood` and `second_mood`.
The UI can show both and let the user pick — much better than forcing a wrong answer.